[<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/> 4. Help Assistant Key Events](https://colab.research.google.com/github/AlbertoLopezCorbalan/MIA-TFM/blob/main/TFM_help_assistant_25_key_events.ipynb)  

# Trabajo Fin de Máster - Help Assistant Key Events By Puzzle
> Autor: Alberto López Corbalán alberto.lopezc@um.es


Con el objetivo de analizar el impacto de cada key event en la predicción del modelo, se utilizará la librería SHAP, que permite estimar la contribución de cada elemento de la secuencia a la salida final del modelo. De esta forma, se podrá interpretar qué eventos tienen mayor peso en la decisión del sistema y cómo influyen en la predicción. Este análisis permitirá identificar los patrones relevantes de la primera parte detectados en experimentaciones anteriores.


Para esta problemática se requiere un entorno con GPU para poder ejecutarse. Basado en el ejemplo de la documentación de SHAP: https://shap.readthedocs.io/en/latest/example_notebooks/text_examples/sentiment_analysis/Emotion%20classification%20multiclass%20example.html



In [ ]:
import sys

if "google.colab" in sys.modules:
    print("Google Colab")
    # Para obtener el dataset si se ejecuta en Colab
    !git clone https://github.com/AlbertoLopezCorbalan/MIA-TFM
    %cd MIA-TFM

Google Colab
fatal: destination path 'MIA-TFM' already exists and is not an empty directory.
/content/MIA-TFM


In [ ]:
import pandas as pd
import re
import numpy as np
import json
import matplotlib.pyplot as plt
import time
import torch
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split, GridSearchCV
from collections import Counter
import shap

# https://huggingface.co/allenai/longformer-base-4096 -> https://arxiv.org/pdf/2004.05150
from datasets import Dataset
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, TrainingArguments, Trainer
from transformers import LongformerTokenizerFast, LongformerForSequenceClassification

print(torch.cuda.get_device_name(0))

GLOBAL_SEED = 123 # Seed global utilizada para garantizar la reproducibilidad
DATASET_PATH = "dataset/complete_report_features_and_text_replay.csv"

Para este problema solo será necesaria la traza, el puzzle y variable objetivo del dataset.


In [ ]:
dataset = pd.read_csv(DATASET_PATH)
print(f"{DATASET_PATH}: " + str(len(dataset)))
dataset.head()

# Preprocesamiento 25%

A continuación, preparamos los datos acordes al experimento. Primero, parseamos los JSON contenidos en la columna attemptFeatures para extraer la variable objetivo `attempt_Completed`. Luego, seleccionamos únicamente la columna de texto `textReplay` junto con la variable objetivo. Finalmente, generamos la versión del DataFrame con un 25% del texto para entrenar y evaluar en qué puntos se centra el modelo, ya que en experimentaciones previas se observó que es en este segmento donde se concentran los patrones más relevantes para la predicción y donde el modelo obtiene los mejores resultados.

In [ ]:
df_llm = dataset.copy(deep = True)

# Parseamos el json
df_llm["attemptFeatures"] = df_llm["attemptFeatures"].apply(json.loads)

# Normalizamos los json para pasarlos a una tabla aplanada
df_attempt = pd.json_normalize(df_llm["attemptFeatures"])

# Quitamos los "." por "_"
df_attempt.columns = df_attempt.columns.str.replace(".", "_", regex=False)

# Unimos todo
df_llm = pd.concat(
    [
        df_llm.drop(columns=["contextFeatures", "globalAttemptId", "attemptFeatures"]),
        df_attempt.add_prefix("attempt_"),
    ],
    axis=1
)

# Seleccionamos solo la textReplay y la variable objetivo
df_llm = df_llm[["textReplay", "attempt_Completed", "puzzle"]]
df_llm["attempt_Completed"] = df_llm["attempt_Completed"].astype("int64")

# Omitimos los prefijos "1. ", "7. " para limpiar la cadena de caracteres
df_llm["puzzle"] = df_llm["puzzle"].apply(lambda x: re.sub(r"^\d+\.\s*", "", x).strip())

# DataFrame al 25%
df_llm_25 = df_llm.copy(deep=True)
df_llm_25["textReplay"] = df_llm_25["textReplay"].apply(lambda x: x[:int(len(x) * 0.25)])

# Entrenamiento

In [ ]:
def train_llm_classifier(df, max_length=4096):

  # Parámetros de configuración
  model_name = "allenai/longformer-base-4096"
  epochs = 1
  train_batch_size = 2
  eval_batch_size = 2
  learning_rate = 2e-5
  weight_decay = 0.01


  # Divide el dataset en entrenamiento y test
  # usando estratificación para mantener proporción de clases
  x_train, x_test, y_train, y_test = train_test_split(df["textReplay"].tolist(), df["attempt_Completed"].tolist(),
                                                      test_size=0.2, random_state=GLOBAL_SEED, stratify=df["attempt_Completed"])

  # Carga el tokenizer del modelo
  tokenizer = LongformerTokenizerFast.from_pretrained(model_name)

  # Se tokeniza del conjunto de entrenamiento y prueba
  train_encodings = tokenizer(x_train, truncation=True, padding=True, max_length=max_length)
  test_encodings = tokenizer(x_test, truncation=True, padding=True, max_length=max_length)

  # Se convierte a formato Dataset de HuggingFace
  train_dataset = Dataset.from_dict({**train_encodings, "labels": y_train})
  test_dataset = Dataset.from_dict({**test_encodings, "labels": y_test})

  # Carga del modelo para clasificación
  model = LongformerForSequenceClassification.from_pretrained(model_name, num_labels=2)

  # Función para calcular rendimiento
  def compute_metrics(eval_pred):

      logits, labels = eval_pred

      # Obtiene la clase con mayor probabilidad
      predictions = np.argmax(logits, axis=-1)

      return {
          "f1": f1_score(labels, predictions),
          "precision": precision_score(labels, predictions),
          "recall": recall_score(labels, predictions),
      }

  # Configuración del entrenamiento
  training_args = TrainingArguments(
      eval_strategy = "epoch",
      logging_strategy = "epoch",
      num_train_epochs = epochs,
      # En estas pruebas no guardaremos el mejor modelo
      load_best_model_at_end = False,
      save_strategy = "no",
      # Batch sizes
      per_device_train_batch_size = train_batch_size,
      per_device_eval_batch_size = eval_batch_size,
      learning_rate = learning_rate,
      weight_decay = weight_decay,
      fp16 = torch.cuda.is_available()
  )

  trainer = Trainer(
      model=model,
      args=training_args,
      train_dataset=train_dataset,
      eval_dataset=test_dataset,
      compute_metrics=compute_metrics,
  )

  trainer.train()

  # Mostramos el rendimiento
  metrics = trainer.evaluate()

  return model, tokenizer, metrics

### Obtener Modelo y Tokenizer Base para Análisis de Importancia de Características

Para poder realizar un análisis de la importancia de las características, necesitamos el modelo entrenado y su tokenizer. Aunque el rendimiento (F1 de ~0.84) es conocido, ejecutaremos la función `train_llm_classifier` con el dataset original (`df_llm_25`) para obtener estos objetos esenciales.

In [ ]:
# Entrenar el modelo con el DataFrame original para obtener el modelo y tokenizer
baseline_model, baseline_tokenizer, baseline_metrics = train_llm_classifier(df_llm_25)
print(f"{baseline_metrics}")

Se muestra el listado de los eventos únicos que estamos manejando

In [ ]:
# Función para extraer eventos desde una cadena de replay en texto
def extract_events(text_replay_string):
    # Expresión regular para encontrar patrones como '(+3.6s) ws-mode_change'
    # Captura la parte del evento después del timestamp
    events = re.findall(r'\(\+\d+\.\d+s\)\s*([a-zA-Z0-9_\-]+)', text_replay_string)
    return events

# Aplicar la función a la columna 'textReplay' de df_llm y obtener todos los eventos únicos
all_events = set()
for text_replay in df_llm['textReplay']:
    extracted_events = extract_events(text_replay)
    for event in extracted_events:
        all_events.add(event)

# Convertir el set a lista ordenada para garantizar consistencia en la salida
unique_events_list = sorted(list(all_events))

print(f"Total unique events found: {len(unique_events_list)}")
print("List of unique events:")
for event in unique_events_list:
    print(f"- {event}")

## Análisis de la frecuencia de eventos

Para identificar los eventos más importantes, vamos a analizar su frecuencia y cómo se distribuyen entre los intentos completados y no completados en el primer tramo de las secuencias, donde se ha detectado el mayor impacto en la predicción. Esto nos puede dar una pista sobre qué eventos tienen un mayor impacto en el resultado final sin necesidad de reentrenar el modelo múltiples veces.

In [ ]:
# Extracción de todos los eventos desde df_llm_25 para análisis de frecuencias
all_extracted_events = []
for text_replay in df_llm_25['textReplay']:
    all_extracted_events.extend(extract_events(text_replay))

event_counts = Counter(all_extracted_events)
total_all_events = sum(event_counts.values())

print("\n--- Top 10 key events ---")
for event, count in event_counts.most_common(10):
    proportion = (count / total_all_events) * 100
    print(f"- {event}: {count} occurrences ({proportion:.2f}%) ")

# Análisis de la distribución de eventos según 'attempt_Completed'
events_completed = []
events_not_completed = []

for index, row in df_llm_25.iterrows():
    events = extract_events(row['textReplay'])
    if row['attempt_Completed'] == 1:  # Se asume 1 = completado, 0 = no completado
        events_completed.extend(events)
    else:
        events_not_completed.extend(events)

completed_event_counts = Counter(events_completed)
not_completed_event_counts = Counter(events_not_completed)

total_completed_events = sum(completed_event_counts.values())
total_not_completed_events = sum(not_completed_event_counts.values())

print("\n--- Most frequent events in COMPLETED attempts ---")
for event, count in completed_event_counts.most_common(10):
    proportion = (count / total_completed_events) * 100
    print(f"- {event}: {count} occurrences ({proportion:.2f}%) ")

print("\n--- Most frequent events in NOT COMPLETED attempts ---")
for event, count in not_completed_event_counts.most_common(10):
    proportion = (count / total_not_completed_events) * 100
    print(f"- {event}: {count} occurrences ({proportion:.2f}%) ")

# SHAP
El objetivo es realizar un análisis con SHAP sobre la columna `textReplay` del DataFrame `df_llm_25` para identificar los eventos más influyentes en la predicción de la finalización de los *puzzles*. Esto implica crear un subconjunto estratificado para SHAP, inicializar un *explainer* de SHAP utilizando el modelo Longformer preentrenado, calcular los valores SHAP, visualizar los resultados y, finalmente, resumir los principales hallazgos obtenidos del análisis.

In [ ]:
_, df_llm_25_test = train_test_split(
    df_llm_25,
    test_size=0.2, # Igual que en el entrenamiento
    stratify=df_llm_25['attempt_Completed'],
    random_state=GLOBAL_SEED
)

# Subconjunto para SHAP
df_llm_25_shap = df_llm_25_test[:10]

df_llm_25_shap.head()

La función `predict_proba_longformer` se encarga de generar las probabilidades de predicción a partir de una lista de textos utilizando un modelo basado en Longformer.



In [ ]:
def predict_proba_longformer(texts):
    # Tokenización de los textos de entrada
    inputs = baseline_tokenizer(list(texts), return_tensors='pt', padding=True, truncation=True, max_length=4096)

    # Mover los inputs al mismo dispositivo que el modelo
    inputs = {k: v.to(baseline_model.device) for k, v in inputs.items()}

    # Obtener las salidas del modelo (logits)
    with torch.no_grad():
        outputs = baseline_model(**inputs)

    # Aplicar softmax para obtener probabilidades
    probabilities = torch.softmax(outputs.logits, dim=1).cpu().numpy()
    return probabilities

Para realizar el análisis de interpretabilidad mediante SHAP, se crea un *masker* específico para datos de texto utilizando el *tokenizer* del modelo Longformer previamente cargado. Este componente permite a SHAP ocultar o reemplazar determinados tokens de la entrada para evaluar cómo afecta su ausencia a la predicción del modelo.

In [ ]:
# Crear un masker para datos de texto usando el tokenizer del modelo base
masker = shap.maskers.Text(
    tokenizer=baseline_tokenizer,
    mask_token=baseline_tokenizer.mask_token,
    collapse_mask_token=True
)

# Inicializar el explicador de SHAP
explainer = shap.Explainer(
    model=predict_proba_longformer,
    masker=masker,
    output_names=['not completed', 'completed']
)

text_samples = df_llm_25_shap['textReplay'].tolist()
shap_values = explainer(text_samples)

Mostramos los resultados de SHAP

In [ ]:
shap.plots.text(shap_values[:, :, "completed"])

# Modificación de `textReplay` y Re-análisis SHAP

Procedemos a modificar la columna `textReplay` para eliminar el texto inicial antes del primer evento `ws-start_level`, basándonos en la observación de shap. Tras esta modificación, volveremos a entrenar el modelo y revisar si cambia el foco del texto

In [ ]:
# Crear una copia profunda del DataFrame original para mantener los datos iniciales sin modificaciones
df_llm_25_modified = df_llm_25.copy(deep=True)

# Función para eliminar todo el texto previo al último evento de inicio del nivel en una traza de replay
def remove_initial_text_until_start_level(text):
    # Buscar todas las apariciones del evento 'ws-start_level'
    matches = list(re.finditer(r'ws-start_level', text))

    if matches:
        # Obtener la posición final de la última aparición del evento 'ws-start_level'
        last_match = matches[-1]
        end_index = last_match.end()

        # Buscar el siguiente separador ' -> ' después del último evento de inicio del nivel
        arrow_index = text.find(' -> ', end_index)

        if arrow_index != -1:
            # Devolver únicamente la parte de la traza posterior al último 'ws-start_level'
            return text[arrow_index + len(' -> '):].strip()
        else:
            # Si no existe el separador, devolver la traza a partir del final del evento 'ws-start_level'
            return text[end_index:].strip()

    # Devolver el texto original si no se encuentra el evento 'ws-start_level'
    return text

# Aplicar la función de preprocesamiento a la columna 'textReplay'
df_llm_25_modified['textReplay'] = df_llm_25_modified['textReplay'].apply(remove_initial_text_until_start_level)

# Mostrar ejemplos comparando las trazas originales y las modificadas
print("Example of modified textReplay (first 5 rows):")
for index, row in df_llm_25_modified.head().iterrows():
    print(f"Original: {df_llm_25.loc[index, 'textReplay']}")
    print(f"Modified: {row['textReplay']}")
    print("\n---\n")

Ahora, entrenaremos nuevamente el modelo utilizando el DataFrame `df_llm_25_modified` para obtener un nuevo modelo y tokenizer que reflejen esta limpieza en las trazas de texto.

In [ ]:
# Entrenar el modelo con el DataFrame modificado para obtener el modelo y tokenizer
baseline_model_modified, baseline_tokenizer_modified, baseline_metrics_modified = train_llm_classifier(df_llm_25_modified)
print(f"{baseline_metrics_modified}")

Creamos un nuevo subconjunto para SHAP a partir del DataFrame modificado y re-inicializamos el explicador con el modelo y tokenizer recién entrenados.

In [ ]:
_, df_llm_25_test_modified = train_test_split(
    df_llm_25_modified,
    test_size=0.2, # Igual que en el entrenamiento
    stratify=df_llm_25_modified['attempt_Completed'],
    random_state=GLOBAL_SEED
)

df_llm_25_shap_modified = df_llm_25_test_modified[:10]


def predict_proba_longformer_modified(texts):
    inputs = baseline_tokenizer_modified(list(texts), return_tensors='pt', padding=True, truncation=True, max_length=4096)
    inputs = {k: v.to(baseline_model_modified.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = baseline_model_modified(**inputs)
    probabilities = torch.softmax(outputs.logits, dim=1).cpu().numpy()
    return probabilities


masker_modified = shap.maskers.Text(
    tokenizer=baseline_tokenizer_modified,
    mask_token=baseline_tokenizer_modified.mask_token,
    collapse_mask_token=True
)

explainer_modified = shap.Explainer(
    model=predict_proba_longformer_modified,
    masker=masker_modified,
    output_names=['not completed', 'completed']
)

### Cálculo y Visualización de Valores SHAP con Texto Modificado

Ahora calcularemos los valores SHAP para el texto modificado y visualizaremos los resultados para observar si hay cambios en la importancia de los eventos.

In [ ]:
text_samples_modified = df_llm_25_shap_modified['textReplay'].tolist()
shap_values_modified = explainer_modified(text_samples_modified)

In [ ]:
shap.plots.text(shap_values_modified[:, :, "completed"])